In [44]:
import numpy as np
import pandas as pd
import os

# --- 1. CONFIGURATION ---
# All files are assumed to be in the local Colab directory (/content/)
file_10k = "P2_8000_probs.npy"
file_18k = "P2_10000_probs.npy"
file_24k = "P2_20000_probs.npy"
test_parquet_file = "test.parquet"

# Ensemble Weights: 8k-0.864 (30%), 10k-0.865 (40%), 20k-0.862  (30%)
weights = [0.35, 0.46, 0.16]

# --- 2. ENSEMBLE CALCULATION ---
print("🚀 Loading .npy probability files from local directory...")
try:
    # Load raw probability arrays from local /content/ directory
    probs_10k = np.load(file_10k)
    probs_18k = np.load(file_18k)
    probs_24k = np.load(file_24k)
except FileNotFoundError as e:
    print(f"❌ Error: {e}. Please ensure all .npy files are uploaded to the /content/ folder.")
    raise

print(f"⚖️ Applying ensemble weights: {weights}")
# Calculate weighted average of probabilities across the three models
# Formula: (P1 * w1) + (P2 * w2) + (P3 * w3)
final_probs = (probs_10k * weights[0]) + (probs_18k * weights[1]) + (probs_24k * weights[2])

# Select the class index with the highest probability (Argmax)
final_preds = np.argmax(final_probs, axis=1)

# --- 3. SUBMISSION GENERATION ---
print("📝 Creating final submission dataframe using local test.parquet...")
try:
    # Load the test file uploaded to the local directory
    test_df = pd.read_parquet(test_parquet_file)
except Exception as e:
    print(f"❌ Error loading {test_parquet_file}: {e}")
    raise

# Construct the final DataFrame with required 'ID' column name
# It checks for 'ID', then 'id', then uses the index as a fallback
submission = pd.DataFrame({
    'ID': test_df.get('ID', test_df.get('id', test_df.index)),
    'label': final_preds
})

# Save the ensemble results to the local Colab directory
output_filename = "submission_ensemble_weighted.csv"
submission.to_csv(output_filename, index=False)

print(f"✅ Ensemble submission saved to: /content/{output_filename}")
print(f"📊 Predicted Class Distribution:\n{submission['label'].value_counts()}")

🚀 Loading .npy probability files from local directory...
⚖️ Applying ensemble weights: [0.44, 0.38, 0.18]
📝 Creating final submission dataframe using local test.parquet...
✅ Ensemble submission saved to: /content/submission_ensemble_weighted.csv
📊 Predicted Class Distribution:
label
0    569
1    232
3    118
2     81
Name: count, dtype: int64
